# Hypotension obs24_target8_gap0 Autoencoder Clusters

Load the prepared data, fitted ViT classifier, and saved explanation maps from the `vit_baseline/obs24_target8_gap0/hypotension` run, then run the autoencoder, embeddings, clustering, and cluster heatmap stages as separate notebook cells.

## Imports and Paths

In [ ]:
from __future__ import annotations

import json
import logging
import re
import shutil
import sys
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "interpretable_ts_vit").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from interpretable_ts_vit.autoencoder import (
    cluster_autoencoder_embeddings,
    create_explanation_value_embeddings,
    train_explanation_value_autoencoder,
)
from interpretable_ts_vit.binning import TimeSeriesBinner
from interpretable_ts_vit.config import ClusterConfig, DataConfig, ExplainConfig, ModelConfig, TrainConfig
from interpretable_ts_vit.data_modules import GenericCSVDataModule
from interpretable_ts_vit.datasets.mimic import configured_variables_for_target, load_mimic_targets_config
from interpretable_ts_vit.io import load_model, load_split
from interpretable_ts_vit.model_modules import ViTTimeSeriesModule
from interpretable_ts_vit.pipeline import _denormalized_patient_value_maps
from interpretable_ts_vit.visualization import aggregate_cluster_value_matrices, patient_value_matrix, plot_value_heatmap, value_ranges_by_variable

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logger = logging.getLogger("hypotension_autoencoder_clusters")
logger.info("Project root: %s", PROJECT_ROOT)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
EXPERIMENT_NAME = "vit_baseline"
TARGET = "hypotension"
WINDOW = "obs24_target8_gap0"
SPLIT = "test"

DATA_ROOT = PROJECT_ROOT / "data" / "mimic_targets"
PROCESSED_DIR = DATA_ROOT / "processed" / WINDOW / TARGET
SOURCE_DIR = DATA_ROOT / WINDOW / TARGET
RUN_DIR = PROJECT_ROOT / "runs" / "mimic_targets" / EXPERIMENT_NAME / WINDOW / TARGET
MIMIC_TARGETS_CONFIG_PATH = PROJECT_ROOT / "configs" / "datasets" / "mimic" / "targets.yaml"

RESET_AUTOENCODER_OUTPUTS = False
RENDER_INSTANCE_HEATMAPS = False

data_config = DataConfig(
    granularity="30min",
    aggregation="mean",
    val_fraction=0.2,
    test_fraction=0.2,
    random_state=13,
)

model_config = ModelConfig(
    patch_size=(1, 4),
    embed_dim=64,
    depth=2,
    num_heads=4,
    mlp_ratio=2.0,
    dropout=0.1,
)

train_config = TrainConfig(
    batch_size=16,
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4,
    device="auto",
    early_stopping_patience=3,
    restore_best_model=True,
    verbose=True,
)

explain_config = ExplainConfig(
    method="grad_attention_rollout",
    target_class=None,
    batch_size=16,
)

cluster_config = ClusterConfig(
    feature_mode="autoencoder",
    method="kmeans",
    n_clusters=8,
    autoencoder_latent_dim=16,
    autoencoder_epochs=50,
    autoencoder_learning_rate=1e-3,
    autoencoder_batch_size=32,
    autoencoder_early_stopping_patience=10,
    plot_mode="value_with_importance_opacity",
    importance_threshold=None,
    show_values=True,
    use_normal_ranges=True,
)

RUN_DIR

## Load Prepared Data, Model, and Explanations

In [ ]:
def allowed_variables_for_target(target: str) -> list[str]:
    mimic_config = load_mimic_targets_config(MIMIC_TARGETS_CONFIG_PATH)
    allowed_variables = configured_variables_for_target(mimic_config, target)
    if not allowed_variables:
        raise ValueError(f"No YAML-configured variables found for target={target!r} in {MIMIC_TARGETS_CONFIG_PATH}")
    return allowed_variables


def count_npy(directory: Path) -> int:
    return sum(1 for _ in directory.glob("*.npy")) if directory.exists() else 0


def require_artifacts(paths: dict[str, Path]) -> None:
    missing = [f"{name}: {path}" for name, path in paths.items() if not path.exists()]
    if missing:
        raise FileNotFoundError("Missing required artifact(s):\n" + "\n".join(missing))


target_data_config = replace(data_config, allowed_variables=allowed_variables_for_target(TARGET))
data = GenericCSVDataModule(
    records_path=SOURCE_DIR / "records.csv",
    labels_path=SOURCE_DIR / "labels.csv",
    processed_dir=PROCESSED_DIR,
    data_config=target_data_config,
)
model = ViTTimeSeriesModule(
    run_dir=RUN_DIR,
    model_config=model_config,
    train_config=train_config,
    explain_config=explain_config,
    cluster_config=cluster_config,
)

required = {
    "source records": SOURCE_DIR / "records.csv",
    "source labels": SOURCE_DIR / "labels.csv",
    "processed binner": PROCESSED_DIR / "binner.json",
    "processed train split": PROCESSED_DIR / "train.npz",
    "processed validation split": PROCESSED_DIR / "val.npz",
    f"processed {SPLIT} split": PROCESSED_DIR / f"{SPLIT}.npz",
    "fitted model": RUN_DIR / "model.pt",
    "run binner": RUN_DIR / "binner.json",
    "run train split": RUN_DIR / "train.npz",
    "run validation split": RUN_DIR / "val.npz",
    f"run {SPLIT} split": RUN_DIR / f"{SPLIT}.npz",
    f"{SPLIT} predictions": model.predictions_path(SPLIT),
    "train explanations": model.explanations_dir("train"),
    "validation explanations": model.explanations_dir("val"),
    f"{SPLIT} explanations": model.explanations_dir(SPLIT),
}
require_artifacts(required)

data.load()
fitted_classifier = load_model(RUN_DIR)
binner = TimeSeriesBinner.load(PROCESSED_DIR / "binner.json")
train_dataset = load_split(PROCESSED_DIR / "train.npz")
val_dataset = load_split(PROCESSED_DIR / "val.npz")
target_dataset = load_split(PROCESSED_DIR / f"{SPLIT}.npz")

train_values = _denormalized_patient_value_maps(train_dataset, binner)
val_values = _denormalized_patient_value_maps(val_dataset, binner)
target_values = _denormalized_patient_value_maps(target_dataset, binner)

artifact_summary = pd.DataFrame(
    [
        {"artifact": "processed", "path": str(PROCESSED_DIR), "patients": None, "files": 3},
        {"artifact": "model", "path": str(RUN_DIR / "model.pt"), "patients": None, "files": 1},
        {"artifact": "train explanations", "path": str(model.explanations_dir("train")), "patients": len(train_values), "files": count_npy(model.explanations_dir("train"))},
        {"artifact": "validation explanations", "path": str(model.explanations_dir("val")), "patients": len(val_values), "files": count_npy(model.explanations_dir("val"))},
        {"artifact": f"{SPLIT} explanations", "path": str(model.explanations_dir(SPLIT)), "patients": len(target_values), "files": count_npy(model.explanations_dir(SPLIT))},
        {"artifact": f"{SPLIT} predictions", "path": str(model.predictions_path(SPLIT)), "patients": len(pd.read_csv(model.predictions_path(SPLIT))), "files": 1},
    ]
)
display(artifact_summary)
print("variables:", len(binner.variable_vocab_))
print("time bins:", len(binner.time_bins_))
print("classifier loaded:", fitted_classifier.__class__.__name__)

## 1. Load or Train the Autoencoder

In [ ]:
clusters_dir = model.clusters_dir(SPLIT)

if RESET_AUTOENCODER_OUTPUTS:
    for path in [
        clusters_dir,
        model.cluster_heatmaps_dir(SPLIT),
        RUN_DIR / "cluster_values" / SPLIT,
        RUN_DIR / "cluster_centroid_heatmaps" / SPLIT,
    ]:
        if path.exists():
            shutil.rmtree(path)

trained_autoencoder = train_explanation_value_autoencoder(
    model.explanations_dir("train"),
    train_values,
    validation_explanations=model.explanations_dir("val"),
    validation_values=val_values,
    output_dir=clusters_dir,
    latent_dim=cluster_config.autoencoder_latent_dim,
    epochs=cluster_config.autoencoder_epochs,
    learning_rate=cluster_config.autoencoder_learning_rate,
    batch_size=cluster_config.autoencoder_batch_size,
    device=train_config.device,
    early_stopping_patience=cluster_config.autoencoder_early_stopping_patience,
    show_progress=True,
    patch_size=model_config.patch_size,
)

autoencoder_metrics = {k: v for k, v in trained_autoencoder["metrics"].items() if k != "history"}
print("Autoencoder artifact:", clusters_dir / "autoencoder.pt")
print("Loaded from cache:", trained_autoencoder.get("loaded_from_cache", False))
display(pd.DataFrame([autoencoder_metrics]))

## 2. Create Embeddings

In [ ]:
embedded_split = create_explanation_value_embeddings(
    model.explanations_dir(SPLIT),
    target_values,
    model=trained_autoencoder["model"],
    preprocessor=trained_autoencoder["preprocessor"],
    output_dir=clusters_dir,
    batch_size=cluster_config.autoencoder_batch_size,
    device=train_config.device,
)

embedding_metrics = {
    "split": SPLIT,
    "n_patients": len(embedded_split["patient_ids"]),
    "embedding_dim": embedded_split["embeddings"].shape[1],
    "reconstruction_loss": embedded_split["loss"],
    "loaded_from_cache": embedded_split.get("loaded_from_cache", False),
}
print("Embedding artifact:", clusters_dir / "autoencoder_embeddings.csv")
display(pd.DataFrame([embedding_metrics]))
display(embedded_split["embedding_frame"].head())

## 3. Cluster Embeddings

In [ ]:
combined_autoencoder_metrics = trained_autoencoder["metrics"] | {"cluster_loss": embedded_split["loss"]}
combined_autoencoder_metadata = {
    **trained_autoencoder["metadata"],
    "cluster": {"n_patients": len(embedded_split["patient_ids"]), **embedded_split["metadata"]},
}

clustered_embeddings = cluster_autoencoder_embeddings(
    embedded_split["embedding_frame"],
    explanations=embedded_split["explanations"],
    predictions=model.predictions_path(SPLIT),
    n_clusters=cluster_config.n_clusters,
    method=cluster_config.method,
    output_dir=clusters_dir,
    autoencoder_metrics=combined_autoencoder_metrics,
    autoencoder_metadata=combined_autoencoder_metadata,
    hdbscan_min_cluster_size=cluster_config.hdbscan_min_cluster_size,
    hdbscan_min_samples=cluster_config.hdbscan_min_samples,
)

print("Cluster artifacts:", clusters_dir)
print("Assignments:", clusters_dir / "cluster_assignments.csv")
display(clustered_embeddings["assignments"].head())
display(clustered_embeddings["assignments"].groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index())

## 4. Plot Clusters

In [ ]:
cluster_heatmaps_dir = model.cluster_heatmaps_dir(SPLIT)
selected_cluster_heatmaps_dir = RUN_DIR / "selected_cluster_heatmaps" / SPLIT
selected_patient_heatmaps_dir = RUN_DIR / "selected_patient_heatmaps" / SPLIT


def safe_path_component(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "class"


def importance_style_from_plot_mode(plot_mode: str):
    if plot_mode == "value_with_importance_opacity":
        return "opacity"
    if plot_mode == "value_with_importance_border":
        return "border"
    if plot_mode == "value":
        return None
    raise ValueError("plot_mode must be 'value', 'value_with_importance_opacity', or 'value_with_importance_border'.")


def normal_ranges_path_for_cluster_config():
    if cluster_config.normal_ranges_path:
        return cluster_config.normal_ranges_path
    if cluster_config.use_normal_ranges:
        return PROJECT_ROOT / "src" / "interpretable_ts_vit" / "normal_ranges.json"
    return None


def plot_selected_cluster(class_label, cluster_number: int) -> Path:
    """Render and display one predicted-class/cluster heatmap."""
    assignments_path = clusters_dir / "cluster_assignments.csv"
    if not assignments_path.exists():
        raise FileNotFoundError(f"Missing cluster assignments: {assignments_path}")

    class_label = str(class_label)
    cluster_number = int(cluster_number)
    assignments = pd.read_csv(assignments_path, dtype={"patient_id": str})
    if "predicted_label" not in assignments.columns:
        raise ValueError("Cluster assignments do not include a predicted_label column.")

    selected_assignments = assignments[
        (assignments["predicted_label"].astype(str) == class_label)
        & (assignments["cluster"].astype(int) == cluster_number)
    ].copy()
    if selected_assignments.empty:
        available = assignments.groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index()
        display(available)
        raise ValueError(f"No patients found for predicted_label={class_label!r}, cluster={cluster_number}.")

    value_matrices = aggregate_cluster_value_matrices(target_dataset, selected_assignments, binner)
    cluster_key = (class_label, cluster_number)
    matrix = value_matrices.get(cluster_key)
    if matrix is None:
        matrix = next(iter(value_matrices.values()))

    importance_style = importance_style_from_plot_mode(cluster_config.plot_mode)
    importance_matrix = None
    if importance_style is not None:
        importance_path = clusters_dir / safe_path_component(class_label) / f"cluster_{cluster_number}.npy"
        if not importance_path.exists():
            importance_path = clusters_dir / f"cluster_{cluster_number}.npy"
        if importance_path.exists():
            importance_matrix = np.load(importance_path)
        else:
            logger.warning("No importance matrix found for %s cluster %s", class_label, cluster_number)

    output_path = selected_cluster_heatmaps_dir / safe_path_component(class_label) / f"cluster_{cluster_number}.png"
    vmin, vmax = value_ranges_by_variable([matrix])
    plot_value_heatmap(
        matrix,
        binner.variable_vocab_,
        binner.time_bins_,
        output_path,
        title=f"Predicted class {class_label}: cluster_{cluster_number} (n={len(selected_assignments)})",
        vmin=vmin,
        vmax=vmax,
        importance_matrix=importance_matrix,
        importance_style=importance_style or "opacity",
        importance_threshold=cluster_config.importance_threshold,
        show_values=cluster_config.show_values,
        normal_ranges=normal_ranges_path_for_cluster_config(),
    )
    print(output_path)
    display(Image(filename=str(output_path)))
    return output_path


def patients_in_cluster(class_label, cluster_number: int) -> pd.DataFrame:
    class_label = str(class_label)
    cluster_number = int(cluster_number)
    assignments_path = clusters_dir / "cluster_assignments.csv"
    assignments = pd.read_csv(assignments_path, dtype={"patient_id": str})
    selected = assignments[
        (assignments["predicted_label"].astype(str) == class_label)
        & (assignments["cluster"].astype(int) == cluster_number)
    ].copy()
    if selected.empty:
        available = assignments.groupby(["predicted_label", "cluster"]).size().rename("patients").reset_index()
        display(available)
        raise ValueError(f"No patients found for predicted_label={class_label!r}, cluster={cluster_number}.")
    return selected.sort_values(["is_centroid", "distance_to_centroid"], ascending=[False, True]).reset_index(drop=True)


def choose_patient_from_cluster(class_label, cluster_number: int, patient_id=None) -> str:
    selected = patients_in_cluster(class_label, cluster_number)
    if patient_id is None:
        centroid_rows = selected[selected["is_centroid"].astype(bool)]
        return str((centroid_rows if not centroid_rows.empty else selected).iloc[0]["patient_id"])
    patient_id = str(patient_id)
    if patient_id not in set(selected["patient_id"].astype(str)):
        display(selected.head(25))
        raise ValueError(f"Patient {patient_id!r} is not in predicted_label={class_label!r}, cluster={cluster_number}.")
    return patient_id


def plot_patient_from_cluster(class_label, cluster_number: int, patient_id=None) -> Path:
    class_label = str(class_label)
    cluster_number = int(cluster_number)
    patient_id = choose_patient_from_cluster(class_label, cluster_number, patient_id)
    matrix = patient_value_matrix(target_dataset, binner, patient_id)
    explanation_path = model.explanations_dir(SPLIT) / f"{patient_id}.npy"
    importance_matrix = np.load(explanation_path) if explanation_path.exists() else None
    if importance_matrix is None:
        logger.warning("No patient explanation found: %s", explanation_path)
    output_path = selected_patient_heatmaps_dir / safe_path_component(class_label) / f"cluster_{cluster_number}" / f"patient_{safe_path_component(patient_id)}.png"
    vmin, vmax = value_ranges_by_variable([matrix])
    importance_style = importance_style_from_plot_mode(cluster_config.plot_mode)
    plot_value_heatmap(
        matrix,
        binner.variable_vocab_,
        binner.time_bins_,
        output_path,
        title=f"Predicted class {class_label}: cluster_{cluster_number}, patient {patient_id}",
        vmin=vmin,
        vmax=vmax,
        importance_matrix=importance_matrix,
        importance_style=importance_style or "opacity",
        importance_threshold=cluster_config.importance_threshold,
        show_values=cluster_config.show_values,
        normal_ranges=normal_ranges_path_for_cluster_config(),
    )
    print(output_path)
    display(Image(filename=str(output_path)))
    return output_path


def plot_cluster_and_patient(class_label, cluster_number: int, patient_id=None) -> dict[str, Path]:
    patient_id = choose_patient_from_cluster(class_label, cluster_number, patient_id)
    return {
        "cluster": plot_selected_cluster(class_label, cluster_number),
        "patient": plot_patient_from_cluster(class_label, cluster_number, patient_id),
    }


# Examples: uncomment and choose an available predicted label / cluster number.
# patients_in_cluster("True", 0).head()
# plot_selected_cluster("True", 0)
# plot_patient_from_cluster("True", 0)  # defaults to the centroid patient
# plot_patient_from_cluster("True", 0, patient_id="31225558")
# plot_cluster_and_patient("True", 0)

## Artifact Check

In [ ]:
important_artifacts = [
    PROCESSED_DIR / "train.npz",
    PROCESSED_DIR / "val.npz",
    PROCESSED_DIR / f"{SPLIT}.npz",
    RUN_DIR / "model.pt",
    model.predictions_path(SPLIT),
    clusters_dir / "autoencoder.pt",
    clusters_dir / "autoencoder_embeddings.csv",
    clusters_dir / "autoencoder_metrics.json",
    clusters_dir / "autoencoder_training_metadata.json",
    clusters_dir / "autoencoder_embedding_metadata.json",
    clusters_dir / "cluster_assignments.csv",
    clusters_dir / "cluster_centroids.csv",
    clusters_dir / "cluster_metadata.json",
    cluster_heatmaps_dir,
    selected_cluster_heatmaps_dir,
    selected_patient_heatmaps_dir,
]

display(pd.DataFrame({"exists": [path.exists() for path in important_artifacts], "path": [str(path) for path in important_artifacts]}))